In [ ]:
import os
import time
import math
import statistics
import traceback
import arcpy
import numpy as np
from arcpy.sa import *
from tqdm import tqdm

# -------------------------------------------------------------------
# WHAT THIS SCRIPT DOES (4 lines)
# -------------------------------------------------------------------
# Fast crown-parameter sweep tuned to KEEP BIG CROWNS while still producing MANY crowns.
# Workflow unchanged: canopy mask → smoothing → local-max surface → seeds → watershed → polygons → optional PAEK smoothing.
# Scores runs using Perimeter (Shape_Length) + Area (Shape_Area) thresholds (P95 + Max), then picks BEST = max n among PASS.
# Writes an HTML report + a GDB results table; uses a robust seed-density gate (coarse resample + numpy) to avoid COUNT/SUM.
# -------------------------------------------------------------------

# -----------------------------------------------------------------------------
# ENVIRONMENT
# -----------------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
arcpy.CheckOutExtension("3D")
arcpy.env.parallelProcessingFactor = "75%"

def stamp(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def exists_or_fail(path: str, what: str):
    if not arcpy.Exists(path):
        raise RuntimeError(f"{what} missing: {path}")

def safe_name(s: str) -> str:
    out = "".join(ch if ch.isalnum() else "_" for ch in s)
    return out[:60]

def raster_minmax(ras_path: str):
    mn = arcpy.management.GetRasterProperties(ras_path, "MINIMUM")[0]
    mx = arcpy.management.GetRasterProperties(ras_path, "MAXIMUM")[0]
    try:
        return float(mn), float(mx)
    except:
        return mn, mx

def percentile(sorted_vals, p: float):
    n = len(sorted_vals)
    if n == 0:
        return 0.0
    if n == 1:
        return float(sorted_vals[0])
    r = (p / 100.0) * (n - 1)
    lo = int(math.floor(r))
    hi = int(math.ceil(r))
    if lo == hi:
        return float(sorted_vals[lo])
    w = r - lo
    return float(sorted_vals[lo] * (1 - w) + sorted_vals[hi] * w)

def summarize(values):
    if not values:
        return {
            "n": 0, "min": 0, "max": 0, "mean": 0, "std": 0,
            "p01": 0, "p05": 0, "p25": 0, "p50": 0, "p75": 0, "p95": 0, "p99": 0
        }
    vals = sorted(values)
    n = len(vals)
    mean = sum(vals) / n
    std = statistics.pstdev(vals) if n > 1 else 0.0
    return {
        "n": n,
        "min": float(vals[0]),
        "max": float(vals[-1]),
        "mean": float(mean),
        "std": float(std),
        "p01": percentile(vals, 1),
        "p05": percentile(vals, 5),
        "p25": percentile(vals, 25),
        "p50": percentile(vals, 50),
        "p75": percentile(vals, 75),
        "p95": percentile(vals, 95),
        "p99": percentile(vals, 99),
    }

def ensure_results_table(gdb: str, table_name: str):
    tpath = os.path.join(gdb, table_name)
    if arcpy.Exists(tpath):
        return tpath
    arcpy.management.CreateTable(gdb, table_name)

    fields = [
        ("CELL","DOUBLE",None),
        ("HMIN","DOUBLE",None),
        ("SMOOTH","LONG",None),
        ("FMAX","LONG",None),
        ("EPS","DOUBLE",None),

        ("SEED_RATIO","DOUBLE",None),

        ("CROWN_N","LONG",None),

        ("MEAN_PERIM_M","DOUBLE",None),
        ("P95_PERIM_M","DOUBLE",None),
        ("MAX_PERIM_M","DOUBLE",None),

        ("MEAN_AREA_M2","DOUBLE",None),
        ("P95_AREA_M2","DOUBLE",None),
        ("MAX_AREA_M2","DOUBLE",None),

        ("PASS","TEXT",10),
        ("SECONDS","DOUBLE",None),
        ("OUT_FC","TEXT",255),
        ("STATUS","TEXT",40),
        ("ERRMSG","TEXT",255),
    ]

    for fn, ft, fl in fields:
        if ft == "TEXT":
            arcpy.management.AddField(tpath, fn, ft, field_length=fl)
        else:
            arcpy.management.AddField(tpath, fn, ft)

    return tpath

def log_result(table_path, row):
    fields = [
        "CELL","HMIN","SMOOTH","FMAX","EPS",
        "SEED_RATIO","CROWN_N",
        "MEAN_PERIM_M","P95_PERIM_M","MAX_PERIM_M",
        "MEAN_AREA_M2","P95_AREA_M2","MAX_AREA_M2",
        "PASS","SECONDS","OUT_FC","STATUS","ERRMSG"
    ]
    with arcpy.da.InsertCursor(table_path, fields) as ic:
        ic.insertRow(row)

def collect_perim_and_area(fc: str, sample_max_features: int = 0):
    """
    Collects Shape_Length (m) and Shape_Area (m²).
    For stable P95/max, keep sample_max_features=0 unless feature count is enormous.
    """
    perims, areas = [], []
    n = int(arcpy.management.GetCount(fc)[0])
    if n == 0:
        return perims, areas

    step = 1
    if sample_max_features and n > sample_max_features:
        step = max(1, n // sample_max_features)

    i = 0
    with arcpy.da.SearchCursor(fc, ["SHAPE@LENGTH", "SHAPE@AREA"]) as cur:
        for (L, A) in cur:
            if step > 1 and (i % step) != 0:
                i += 1
                continue
            perims.append(float(L))
            areas.append(float(A))
            i += 1
    return perims, areas

def write_html_report(html_path, meta, run_rows, best_row=None, pass_row=None):
    def esc(s):
        s = "" if s is None else str(s)
        return (s.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
                 .replace('"',"&quot;").replace("'","&#39;"))
    def fmt(x, nd=2):
        try:
            return f"{float(x):.{nd}f}"
        except:
            return esc(x)

    lines = []
    lines.append("<!DOCTYPE html><html><head><meta charset='utf-8'>")
    lines.append("<title>Crown sweep report (big crowns + many crowns)</title>")
    lines.append("""
<style>
body{font-family:Arial,Helvetica,sans-serif;margin:20px;line-height:1.35}
.small{color:#555;font-size:0.95em}
table{border-collapse:collapse;width:100%;margin:12px 0}
th,td{border:1px solid #ddd;padding:6px 8px;font-size:0.92em;vertical-align:top}
th{background:#f6f6f6;text-align:left}
.good{background:#ecffef}
.bad{background:#ffecec}
.mono{font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace}
</style>
""")
    lines.append("</head><body>")
    lines.append("<h1>Crown parameter sweep report</h1>")
    lines.append("<h3>Goal: big crowns + still many crowns (PASS uses P95+Max; BEST = max n among PASS)</h3>")
    lines.append(f"<div class='small'>Generated: {esc(meta['generated_at'])}</div>")

    lines.append("<h2>Inputs & thresholds</h2><table>")
    for k in meta.keys():
        lines.append(f"<tr><th>{esc(k)}</th><td class='mono'>{esc(meta.get(k))}</td></tr>")
    lines.append("</table>")

    lines.append("<h2>Best run (overall)</h2>")
    if best_row:
        lines.append("<table>")
        lines.append(f"<tr><th>Params</th><td class='mono'>CELL={best_row['CELL']}, HMIN={best_row['HMIN']}, SMOOTH={best_row['SMOOTH']}, FMAX={best_row['FMAX']}, EPS={best_row['EPS']}</td></tr>")
        lines.append(f"<tr><th>Status</th><td>{esc(best_row['status'])}</td></tr>")
        lines.append(f"<tr><th>PASS?</th><td>{esc(best_row['pass_flag'])}</td></tr>")
        lines.append(f"<tr><th>n</th><td>{best_row['n']:,}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Perimeter (m)</th><td>{fmt(best_row['perim']['p95'])} / {fmt(best_row['perim']['max'])}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Area (m²)</th><td>{fmt(best_row['area']['p95'])} / {fmt(best_row['area']['max'])}</td></tr>")
        lines.append(f"<tr><th>Output FC</th><td class='mono'>{esc(best_row['out_fc'])}</td></tr>")
        lines.append("</table>")
    else:
        lines.append("<p>No successful run produced crowns.</p>")

    if pass_row:
        lines.append("<h2>Best PASS run (max n among PASS)</h2>")
        lines.append("<table>")
        lines.append(f"<tr><th>Params</th><td class='mono'>CELL={pass_row['CELL']}, HMIN={pass_row['HMIN']}, SMOOTH={pass_row['SMOOTH']}, FMAX={pass_row['FMAX']}, EPS={pass_row['EPS']}</td></tr>")
        lines.append(f"<tr><th>n</th><td>{pass_row['n']:,}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Perimeter (m)</th><td>{fmt(pass_row['perim']['p95'])} / {fmt(pass_row['perim']['max'])}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Area (m²)</th><td>{fmt(pass_row['area']['p95'])} / {fmt(pass_row['area']['max'])}</td></tr>")
        lines.append(f"<tr><th>Output FC</th><td class='mono'>{esc(pass_row['out_fc'])}</td></tr>")
        lines.append("</table>")

    lines.append("<h2>All runs</h2>")
    lines.append("<div class='small'>Green rows PASS. BEST(PASS) = maximum n. ERRMSG truncated.</div>")
    lines.append("<table>")
    headers = [
        "CELL","HMIN","SMOOTH","FMAX","EPS","status","seconds","seed_ratio",
        "PASS","n",
        "perim_mean","perim_p95","perim_max",
        "area_mean","area_p95","area_max",
        "out_fc","errmsg"
    ]
    lines.append("<tr>" + "".join(f"<th>{h}</th>" for h in headers) + "</tr>")
    for r in run_rows:
        cls = "good" if r.get("pass_flag") == "PASS" else ("bad" if r.get("status","").startswith("ERROR") else "")
        lines.append(f"<tr class='{cls}'>" + "".join([
            f"<td>{esc(r.get('CELL'))}</td>",
            f"<td>{esc(r.get('HMIN'))}</td>",
            f"<td>{esc(r.get('SMOOTH'))}</td>",
            f"<td>{esc(r.get('FMAX'))}</td>",
            f"<td>{esc(r.get('EPS'))}</td>",
            f"<td>{esc(r.get('status'))}</td>",
            f"<td>{fmt(r.get('seconds',0),1)}</td>",
            f"<td>{fmt(r.get('seed_ratio',0),6)}</td>",
            f"<td>{esc(r.get('pass_flag',''))}</td>",
            f"<td>{int(r.get('n',0)):,}</td>",
            f"<td>{fmt(r.get('perim',{}).get('mean',0))}</td>",
            f"<td>{fmt(r.get('perim',{}).get('p95',0))}</td>",
            f"<td>{fmt(r.get('perim',{}).get('max',0))}</td>",
            f"<td>{fmt(r.get('area',{}).get('mean',0))}</td>",
            f"<td>{fmt(r.get('area',{}).get('p95',0))}</td>",
            f"<td>{fmt(r.get('area',{}).get('max',0))}</td>",
            f"<td class='mono'>{esc(r.get('out_fc',''))}</td>",
            f"<td class='mono'>{esc(r.get('errmsg',''))}</td>",
        ]) + "</tr>")
    lines.append("</table></body></html>")

    os.makedirs(os.path.dirname(html_path), exist_ok=True)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

# -----------------------------------------------------------------------------
# INPUTS
# -----------------------------------------------------------------------------
LASD = r"C:\ArcProj\AboveGroundBiomass\Wait_LasDataset.lasd"
GDB  = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb"
CHM  = os.path.join(GDB, "chm_raw_CopyRaster")
exists_or_fail(CHM, "CHM raster")

# -----------------------------------------------------------------------------
# AOI (HUGE SPEED WIN) — set to None to disable
# -----------------------------------------------------------------------------
AOI = None

# -----------------------------------------------------------------------------
# PASS THRESHOLDS (Big crowns + still many crowns)
# -----------------------------------------------------------------------------
# Robust thresholds (recommended): use P95 + Max rather than mean.
P95_PERIM_MAX = 80.0     # m
MAX_PERIM_MAX = 150.0    # m
P95_AREA_MAX  = 300.0    # m²
MAX_AREA_MAX  = 700.0    # m²

def pass_rules(perim_stats, area_stats):
    return (
        perim_stats["p95"] <= P95_PERIM_MAX and
        perim_stats["max"] <= MAX_PERIM_MAX and
        area_stats["p95"]  <= P95_AREA_MAX and
        area_stats["max"]  <= MAX_AREA_MAX
    )

# -----------------------------------------------------------------------------
# SEED DENSITY GATE (approx, cheap early reject)
# -----------------------------------------------------------------------------
SEED_RATIO_MIN = 0.00005
SEED_RATIO_MAX = 0.05
GATE_RES_M = 10

# -----------------------------------------------------------------------------
# OUTPUT / PERFORMANCE SETTINGS
# -----------------------------------------------------------------------------
KEEP_ALL_OUTPUTS = False
DO_POLY_SMOOTH = True
PAEK_TOL = "3 Meters"

# For accurate max & P95, keep this 0 (exact). If too slow, set e.g. 50000.
SAMPLE_MAX_FEATURES = 0

REPORT_DIR = r"C:\ArcProj\AboveGroundBiomass\reports"
REPORT_HTML = os.path.join(REPORT_DIR, f"crown_sweep_big_many_{time.strftime('%Y%m%d_%H%M%S')}.html")

RESULTS_TABLE_NAME = safe_name(f"crown_sweep_big_many_{time.strftime('%Y%m%d_%H%M%S')}")
RESULTS_TABLE = ensure_results_table(GDB, RESULTS_TABLE_NAME)

# -----------------------------------------------------------------------------
# PARAMETER SETS (small + targeted to keep bigger crowns, but still allow many)
# -----------------------------------------------------------------------------
# Notes:
# - FMAX up suppresses micro-peaks (reduces fragmentation, keeps bigger crowns)
# - EPS moderate-to-higher reduces over-splitting driven by tiny height differences
# - SMOOTH 1–3 reduces micro-basins while still keeping structure
# Keep this list short for <30 minutes.
PARAM_SETS = [
    #  current best (baseline)
    (1, 1.3, 3,11, 0.8),

    # Capture more crown edge (lower HMIN)
    (1, 1.3, 2, 10, 0.6),
    (1, 1.3, 2, 10, 0.7),

    # Lower HMIN but reduce micro-splitting
    (1, 1.1, 3, 10, 0.6),
    (1, 1.1, 3, 10, 0.7),

    # Lower HMIN + slightly more merging tolerance
    (1, 1.1, 2, 11, 0.6),
    (1, 1.1, 2, 11, 0.7),
    (1, 1.1, 2, 10, 0.6),
    (1, 1.1, 2, 10, 0.7)
]


# -----------------------------------------------------------------------------
# PRE-FLIGHT: CHM sanity check
# -----------------------------------------------------------------------------
mn, mx = raster_minmax(CHM)
stamp(f"CHM range: min={mn}, max={mx}")
if isinstance(mx, (int, float)) and mx > 80:
    stamp("WARNING: CHM max > 80 — double-check this is a true CHM (DSM-DTM).")

# -----------------------------------------------------------------------------
# AOI speed-up
# -----------------------------------------------------------------------------
if AOI:
    exists_or_fail(AOI, "AOI polygon")
    arcpy.env.extent = AOI
    arcpy.env.mask = AOI
    stamp(f"AOI enabled: {AOI}")
else:
    arcpy.env.extent = None
    arcpy.env.mask = None

# -----------------------------------------------------------------------------
# CHM resample cache (per CELL)
# -----------------------------------------------------------------------------
resample_cache = {}

def get_chm_for_cell(cell_size: float):
    try:
        csx = float(arcpy.management.GetRasterProperties(CHM, "CELLSIZEX")[0])
    except:
        csx = None

    if csx is not None and abs(float(cell_size) - csx) < 1e-6:
        return CHM

    if cell_size in resample_cache and arcpy.Exists(resample_cache[cell_size]):
        return resample_cache[cell_size]

    out_ras = os.path.join(GDB, safe_name(f"CHM_cell{cell_size}m"))
    if arcpy.Exists(out_ras):
        resample_cache[cell_size] = out_ras
        return out_ras

    stamp(f"Resampling CHM to {cell_size} m...")
    arcpy.management.Resample(CHM, out_ras, f"{cell_size} {cell_size}", "BILINEAR")
    exists_or_fail(out_ras, f"CHM resampled {cell_size}m")
    resample_cache[cell_size] = out_ras
    return out_ras

# -----------------------------------------------------------------------------
# Raster caches (avoid recomputing heavy rasters)
# -----------------------------------------------------------------------------
cache_canopy = {}   # (CELL,HMIN) -> raster path
cache_smooth = {}   # (CELL,HMIN,SMOOTH) -> raster path
cache_fmax   = {}   # (CELL,HMIN,SMOOTH,FMAX) -> raster path

# -----------------------------------------------------------------------------
# Seed gate (robust approx using coarse resample + numpy)
# -----------------------------------------------------------------------------
def approx_seed_ratio(seed_ras: str, canopy_ras: str, tag: str):
    seed_coarse = os.path.join(GDB, safe_name(f"seed_gate_{tag}"))
    cany_coarse = os.path.join(GDB, safe_name(f"can_gate_{tag}"))

    for p in [seed_coarse, cany_coarse]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    arcpy.management.Resample(seed_ras, seed_coarse, f"{GATE_RES_M} {GATE_RES_M}", "NEAREST")
    arcpy.management.Resample(canopy_ras, cany_coarse, f"{GATE_RES_M} {GATE_RES_M}", "NEAREST")
    exists_or_fail(seed_coarse, "seed_coarse")
    exists_or_fail(cany_coarse, "canopy_coarse")

    seed_r = arcpy.Raster(seed_coarse)
    cany_r = arcpy.Raster(cany_coarse)

    seed_nodata = seed_r.noDataValue
    cany_nodata = cany_r.noDataValue

    a_seed = arcpy.RasterToNumPyArray(seed_r, nodata_to_value=seed_nodata)
    a_cany = arcpy.RasterToNumPyArray(cany_r, nodata_to_value=cany_nodata)

    canopy_cells = int(np.count_nonzero(a_cany != cany_nodata))
    seed_cells   = int(np.count_nonzero(a_seed == 1))

    for p in [seed_coarse, cany_coarse]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    return seed_cells / max(canopy_cells, 1)

# -----------------------------------------------------------------------------
# Run bookkeeping and best-selection logic
# -----------------------------------------------------------------------------
run_rows = []
best_row = None       # best overall (PASS wins; otherwise closest by penalty)
best_pass_row = None  # BEST PASS for your goal: MAX n among PASS (then lower P95 tail)

def penalty(perim_stats, area_stats):
    # Lower is better; 0 means PASS.
    def over(x, lim): return max(0.0, x - lim)
    return (
        over(perim_stats["p95"], P95_PERIM_MAX) * 2.0 +
        over(perim_stats["max"], MAX_PERIM_MAX) * 3.0 +
        over(area_stats["p95"],  P95_AREA_MAX)  * 2.0 +
        over(area_stats["max"],  MAX_AREA_MAX)  * 3.0
    )

def is_better_pass(a, b):
    """
    PASS ranking for goal = big crowns + still many crowns:
    1) maximize n
    2) minimize P95 perimeter (less blob tail)
    3) minimize MAX perimeter
    4) minimize P95 area
    """
    if b is None:
        return True
    if a["n"] != b["n"]:
        return a["n"] > b["n"]
    if a["perim"]["p95"] != b["perim"]["p95"]:
        return a["perim"]["p95"] < b["perim"]["p95"]
    if a["perim"]["max"] != b["perim"]["max"]:
        return a["perim"]["max"] < b["perim"]["max"]
    return a["area"]["p95"] < b["area"]["p95"]

# -----------------------------------------------------------------------------
# One combo runner
# -----------------------------------------------------------------------------
def run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, tag="RUN"):
    t0 = time.time()
    status = "OK"
    out_fc = ""
    seed_ratio = 0.0
    errmsg = ""

    CHM_BASE = get_chm_for_cell(CELL)

    # 1) CANOPY (cached)
    k1 = (CELL, HMIN)
    CHM_CANOPY = cache_canopy.get(k1)
    if not CHM_CANOPY or not arcpy.Exists(CHM_CANOPY):
        CHM_CANOPY = os.path.join(GDB, safe_name(f"CHM_CANOPY_c{CELL}_h{HMIN}"))
        if arcpy.Exists(CHM_CANOPY):
            arcpy.management.Delete(CHM_CANOPY)
        SetNull(Raster(CHM_BASE) < HMIN, Raster(CHM_BASE)).save(CHM_CANOPY)
        exists_or_fail(CHM_CANOPY, "CHM_CANOPY")
        cache_canopy[k1] = CHM_CANOPY

    # Lock env to canopy only (stability + speed)
    arcpy.env.snapRaster = CHM_BASE
    arcpy.env.cellSize   = CHM_BASE
    arcpy.env.extent     = CHM_CANOPY
    arcpy.env.mask       = CHM_CANOPY

    # 2) SMOOTH (cached)
    k2 = (CELL, HMIN, SMOOTH)
    CHM_S = cache_smooth.get(k2)
    if not CHM_S or not arcpy.Exists(CHM_S):
        CHM_S = os.path.join(GDB, safe_name(f"CHM_S_c{CELL}_h{HMIN}_s{SMOOTH}"))
        if arcpy.Exists(CHM_S):
            arcpy.management.Delete(CHM_S)
        if SMOOTH == 0:
            Raster(CHM_CANOPY).save(CHM_S)
        else:
            FocalStatistics(Raster(CHM_CANOPY), NbrCircle(SMOOTH, "CELL"), "MEAN", "DATA").save(CHM_S)
        exists_or_fail(CHM_S, "CHM_S")
        cache_smooth[k2] = CHM_S

    # 3) Local max surface (cached)
    k3 = (CELL, HMIN, SMOOTH, FMAX)
    CHM_F = cache_fmax.get(k3)
    if not CHM_F or not arcpy.Exists(CHM_F):
        CHM_F = os.path.join(GDB, safe_name(f"CHM_FMAX_c{CELL}_h{HMIN}_s{SMOOTH}_f{FMAX}"))
        if arcpy.Exists(CHM_F):
            arcpy.management.Delete(CHM_F)
        FocalStatistics(Raster(CHM_S), NbrCircle(FMAX, "CELL"), "MAXIMUM", "DATA").save(CHM_F)
        exists_or_fail(CHM_F, "CHM_FMAX")
        cache_fmax[k3] = CHM_F

    combo_tag = safe_name(f"{tag}_c{CELL}_h{HMIN}_s{SMOOTH}_f{FMAX}_e{EPS}")
    SEEDS_RAS  = os.path.join(GDB, safe_name(f"SEEDS_{combo_tag}"))
    WS_RAS     = os.path.join(GDB, safe_name(f"WS_{combo_tag}"))
    WS_CLEAN   = os.path.join(GDB, safe_name(f"WSCL_{combo_tag}"))
    CROWNS_RAW = os.path.join(GDB, safe_name(f"CROWNS_{combo_tag}"))
    CROWNS_FIN = os.path.join(GDB, safe_name(f"CROWNSF_{combo_tag}"))

    for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW, CROWNS_FIN]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    try:
        # 4) Seeds (Thin OFF by design)
        SEEDS = Con(
            (Raster(CHM_S) >= (Raster(CHM_F) - EPS)) &
            (Raster(CHM_S) >= HMIN),
            1
        )
        SEEDS.save(SEEDS_RAS)
        exists_or_fail(SEEDS_RAS, "SEEDS_RAS")

        # Seed gate
        seed_ratio = approx_seed_ratio(SEEDS_RAS, CHM_CANOPY, tag=combo_tag)
        if seed_ratio < SEED_RATIO_MIN:
            status = "SKIP_TOO_FEW_SEEDS"
            raise RuntimeError(status)
        if seed_ratio > SEED_RATIO_MAX:
            status = "SKIP_TOO_MANY_SEEDS"
            raise RuntimeError(status)

        # 5) Watershed
        INV = -1 * Raster(CHM_S)
        FDR = FlowDirection(INV, "FORCE")
        Watershed(FDR, SEEDS_RAS).save(WS_RAS)
        exists_or_fail(WS_RAS, "WS_RAS")

        # 5b) Reduce tiny zones before polygonizing
        MajorityFilter(Raster(WS_RAS), "EIGHT", "HALF").save(WS_CLEAN)
        exists_or_fail(WS_CLEAN, "WS_CLEAN")

        # 6) Polygonize
        arcpy.conversion.RasterToPolygon(
            in_raster=WS_CLEAN,
            out_polygon_features=CROWNS_RAW,
            simplify="SIMPLIFY",
            raster_field="Value",
            create_multipart_features="SINGLE_OUTER_PART"
        )
        exists_or_fail(CROWNS_RAW, "CROWNS_RAW")

        # 7) Optional smoothing
        if DO_POLY_SMOOTH:
            arcpy.cartography.SmoothPolygon(
                in_features=CROWNS_RAW,
                out_feature_class=CROWNS_FIN,
                algorithm="PAEK",
                tolerance=PAEK_TOL
            )
            exists_or_fail(CROWNS_FIN, "CROWNS_FIN")
            out_fc = CROWNS_FIN
        else:
            out_fc = CROWNS_RAW

        # 8) Stats
        n = int(arcpy.management.GetCount(out_fc)[0])
        if n == 0:
            status = "FAIL_EMPTY"
            perims, areas = [], []
        else:
            perims, areas = collect_perim_and_area(out_fc, sample_max_features=SAMPLE_MAX_FEATURES)

        perim_stats = summarize(perims)
        area_stats  = summarize(areas)
        dt = time.time() - t0

        pass_flag = "PASS" if (n > 0 and pass_rules(perim_stats, area_stats)) else "FAIL"
        pen = penalty(perim_stats, area_stats) if n > 0 else 1e9

        row = {
            "CELL": CELL, "HMIN": HMIN, "SMOOTH": SMOOTH, "FMAX": FMAX, "EPS": EPS,
            "status": status, "seconds": dt,
            "seed_ratio": seed_ratio, "errmsg": "",
            "n": int(n),
            "perim": perim_stats,
            "area": area_stats,
            "pass_flag": pass_flag,
            "penalty": pen,
            "out_fc": out_fc
        }

        log_result(
            RESULTS_TABLE,
            (CELL, HMIN, SMOOTH, FMAX, EPS,
             float(seed_ratio),
             int(n),
             float(perim_stats["mean"]), float(perim_stats["p95"]), float(perim_stats["max"]),
             float(area_stats["mean"]),  float(area_stats["p95"]),  float(area_stats["max"]),
             pass_flag,
             float(dt),
             out_fc,
             status,
             "")
        )

        if not KEEP_ALL_OUTPUTS:
            for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW]:
                if arcpy.Exists(p) and p != out_fc:
                    arcpy.management.Delete(p)

        return row

    except Exception as e:
        dt = time.time() - t0
        if not status.startswith("SKIP"):
            status = "ERROR"
        errmsg = (str(e) or "unknown error")[:255]

        try:
            log_result(
                RESULTS_TABLE,
                (CELL, HMIN, SMOOTH, FMAX, EPS,
                 float(seed_ratio),
                 0,
                 0.0, 0.0, 0.0,
                 0.0, 0.0, 0.0,
                 "FAIL",
                 float(dt),
                 "",
                 status,
                 errmsg)
            )
        except:
            pass

        for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW, CROWNS_FIN]:
            if arcpy.Exists(p):
                arcpy.management.Delete(p)

        return {
            "CELL": CELL, "HMIN": HMIN, "SMOOTH": SMOOTH, "FMAX": FMAX, "EPS": EPS,
            "status": status, "seconds": dt,
            "seed_ratio": seed_ratio, "errmsg": errmsg,
            "n": 0,
            "perim": summarize([]),
            "area": summarize([]),
            "pass_flag": "FAIL",
            "penalty": 1e9,
            "out_fc": ""
        }

# -----------------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------------
try:
    stamp(f"Running {len(PARAM_SETS)} targeted combos (big crowns + many crowns)...")

    for (CELL, HMIN, SMOOTH, FMAX, EPS) in tqdm(PARAM_SETS, desc="Sweep", unit="combo"):
        row = run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, tag="LAST_1m")
        run_rows.append(row)

        # Track best PASS: max n among PASS, then smaller tail
        if row["status"] == "OK" and row["pass_flag"] == "PASS" and row["n"] > 0:
            if is_better_pass(row, best_pass_row):
                best_pass_row = row

        # Track best overall if nothing passes: minimum penalty
        if row["status"] == "OK" and row["n"] > 0:
            if (best_row is None) or (row["penalty"] < best_row["penalty"]):
                best_row = row

except Exception:
    stamp("FATAL ERROR (full traceback):")
    print(traceback.format_exc())

finally:
    winner = best_pass_row if best_pass_row is not None else best_row

    meta = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "GDB": GDB,
        "CHM": CHM,
        "AOI": AOI,
        "P95_PERIM_MAX": P95_PERIM_MAX,
        "MAX_PERIM_MAX": MAX_PERIM_MAX,
        "P95_AREA_MAX": P95_AREA_MAX,
        "MAX_AREA_MAX": MAX_AREA_MAX,
        "SEED_RATIO_MIN": SEED_RATIO_MIN,
        "SEED_RATIO_MAX": SEED_RATIO_MAX,
        "GATE_RES_M": GATE_RES_M,
        "DO_POLY_SMOOTH": DO_POLY_SMOOTH,
        "PAEK_TOL": PAEK_TOL,
        "KEEP_ALL_OUTPUTS": KEEP_ALL_OUTPUTS,
        "SAMPLE_MAX_FEATURES": SAMPLE_MAX_FEATURES,
        "RESULTS_TABLE": RESULTS_TABLE,
        "PARAM_SETS": PARAM_SETS,
        "BEST_RULE": "BEST(PASS)=max n, then min P95 perimeter, then min max perimeter, then min P95 area"
    }

    stamp(f"Writing HTML report: {REPORT_HTML}")
    write_html_report(REPORT_HTML, meta, run_rows, best_row=winner, pass_row=best_pass_row)
    stamp(f"✔ Report written: {REPORT_HTML}")
    stamp(f"Results table: {RESULTS_TABLE}")

    if winner:
        stamp("WINNER:")
        stamp(f"  PASS? {winner['pass_flag']} | status={winner['status']}")
        stamp(f"  Params: CELL={winner['CELL']}, HMIN={winner['HMIN']}, SMOOTH={winner['SMOOTH']}, FMAX={winner['FMAX']}, EPS={winner['EPS']}")
        stamp(f"  n={winner['n']:,} | seed_ratio≈{winner.get('seed_ratio',0):.6f}")
        stamp(f"  Perim mean/p95/max (m): {winner['perim']['mean']:.2f} / {winner['perim']['p95']:.2f} / {winner['perim']['max']:.2f}")
        stamp(f"  Area  mean/p95/max (m²): {winner['area']['mean']:.2f} / {winner['area']['p95']:.2f} / {winner['area']['max']:.2f}")
        stamp(f"  Output FC: {winner['out_fc']}")
    else:
        stamp("No successful crowns were produced. Check ERRMSG in the HTML/table.")

    try:
        import winsound
        winsound.Beep(880, 500)
    except:
        pass
